In [21]:
import os

In [25]:
os.chdir("./Desktop/medical-dcgan")

In [26]:
%pwd

'c:\\Users\\asdaw\\Desktop\\medical-dcgan'

In [27]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataPreprocessingConfig:
    root_dir: Path
    data_path: Path
    image_size: int
    batch_size: int
    num_workers: int
    pin_memory: bool


In [28]:
from dcGAN_image_generation.constants import *
from dcGAN_image_generation.utils.common import read_yaml, create_directories, save_json

In [29]:

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        self.artifacts_root = ROOT_DIR / self.config.artifacts_root
        create_directories([self.artifacts_root])
        
    def get_data_preprocessing_config(self) -> DataPreprocessingConfig:

        config = self.config.data_preprocessing
    
        create_directories([config.root_dir])
    
        data_preprocessing_config = DataPreprocessingConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            image_size=config.image_size,
            batch_size=config.batch_size,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory
        )
    
        return data_preprocessing_config


In [30]:
import os
import json
from pathlib import Path
from dcGAN_image_generation import logger
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [31]:

class DataPreprocessing:
    def __init__(self, config: DataPreprocessingConfig):
        self.config = config

    def _get_transforms(self):
        logger.info("Creating DCGAN-compatible transforms")

        transform = transforms.Compose([
            transforms.Resize(self.config.image_size),
            transforms.CenterCrop(self.config.image_size),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5],
                                 [0.5, 0.5, 0.5])
            ])

        return transform

    def _load_dataset(self):
        logger.info("Loading dataset from ingestion artifacts")

        dataset = datasets.ImageFolder(
            root=self.config.data_path / "train",
            transform=self._get_transforms()
        )

        logger.info(f"Dataset loaded with {len(dataset)} images")

        return dataset

    def save_metadata(self) -> None:
        """
        Saves dataset metadata for reproducibility and pipeline tracking
        """

        try:
            dataset = self._load_dataset()

            metadata = {
                "dataset_size": len(dataset),
                "image_size": self.config.image_size,
                "batch_size": self.config.batch_size
            }

            metadata_path = Path(self.config.root_dir) / "metadata.json"
            metadata_path.parent.mkdir(parents=True, exist_ok=True)

            with open(metadata_path, "w") as f:
                json.dump(metadata, f, indent=4)

            logger.info(f"Metadata saved at {metadata_path}")

        except Exception as e:
            raise e
    
    def validate_batch(self) -> None:
        """
        Validates one batch for:
        - shape
        - pixel range
        """

        try:
            dataset = self._load_dataset()

            dataloader = DataLoader(
                dataset,
                batch_size=self.config.batch_size,
                shuffle=True,
                num_workers=2,  # safe for Windows
                pin_memory=self.config.pin_memory
            )

            images, _ = next(iter(dataloader))

            logger.info(f"Batch shape: {images.shape}")
            logger.info(f"Pixel min: {images.min().item()}")
            logger.info(f"Pixel max: {images.max().item()}")

            if images.shape[1:] != (3, self.config.image_size, self.config.image_size):
                raise ValueError("Image shape mismatch for DCGAN")

            if images.min() < -1.1 or images.max() > 1.1:
                raise ValueError("Pixel range not in [-1, 1]")

            logger.info("Batch validation successful")

        except Exception as e:
            raise e



In [32]:

try:
    config = ConfigurationManager()

    data_preprocessing_config = config.get_data_preprocessing_config()

    data_preprocessing = DataPreprocessing(config=data_preprocessing_config)

    data_preprocessing.save_metadata()
    data_preprocessing.validate_batch()

except Exception as e:
    raise e

[2026-02-19 09:45:45,662: INFO: common]: yaml file loaded: config\config.yaml]
[2026-02-19 09:45:45,665: INFO: common]: yaml file loaded: params.yaml]
[2026-02-19 09:45:45,666: INFO: common]: created directory at: C:\Users\asdaw\Desktop\medical-dcgan\artifacts]
[2026-02-19 09:45:45,667: INFO: common]: created directory at: artifacts/data_preprocessing]
[2026-02-19 09:45:45,668: INFO: 3495630993]: Loading dataset from ingestion artifacts]
[2026-02-19 09:45:45,668: INFO: 3495630993]: Creating DCGAN-compatible transforms]
[2026-02-19 09:45:45,699: INFO: 3495630993]: Dataset loaded with 5216 images]
[2026-02-19 09:45:45,701: INFO: 3495630993]: Metadata saved at artifacts\data_preprocessing\metadata.json]
[2026-02-19 09:45:45,702: INFO: 3495630993]: Loading dataset from ingestion artifacts]
[2026-02-19 09:45:45,702: INFO: 3495630993]: Creating DCGAN-compatible transforms]
[2026-02-19 09:45:45,730: INFO: 3495630993]: Dataset loaded with 5216 images]
[2026-02-19 09:45:55,411: INFO: 3495630993